In [2]:
import os
import torch
import torchvision
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision import transforms as T
from PIL import Image
import xml.etree.ElementTree as ET
from tqdm.auto import tqdm

In [3]:
CONFIG = {
    "data_dir": "TrashBox_train_set",
    "num_classes": 8,     # background (0) + your classes
    "image_size": 224,
    "batch_size": 1,
    "num_epochs": 10,
    "learning_rate": 1e-3,
    "val_split": 0.15,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "seed": 42
}

In [4]:
from torchvision import datasets

dataset = datasets.ImageFolder(CONFIG["data_dir"])

print(dataset.classes)
print(len(dataset.classes))
print(dataset.class_to_idx)

['cardboard', 'e-waste', 'glass', 'medical', 'metal', 'paper', 'plastic']
7
{'cardboard': 0, 'e-waste': 1, 'glass': 2, 'medical': 3, 'metal': 4, 'paper': 5, 'plastic': 6}


In [5]:
# Mapping the class names to integer IDs (0 = background, always required)
LABEL_MAP = {
    "background": 0,
    "cardboard":    1,
    "e-waste":    2,
    "glass":    3,
    "medical":    4,
    "metal":    5,
    "paper":    6,
    "plastic":    7
}

In [6]:
OUTPUT_JSON = os.path.join(CONFIG["data_dir"], "annotations.json")
IMG_EXTS    = (".jpg", ".jpeg", ".png", ".bmp")

In [7]:
categories = []
images     = []
annotations = []

In [8]:
# Collect class names from subfolder names (sorted for consistency)
class_names = sorted([
    d for d in os.listdir(CONFIG["data_dir"])
    if os.path.isdir(os.path.join(CONFIG["data_dir"], d))
])
 
print(f"Found {len(class_names)} classes: {class_names}")

Found 7 classes: ['cardboard', 'e-waste', 'glass', 'medical', 'metal', 'paper', 'plastic']


In [9]:
for cat_id, name in enumerate(class_names, start=1):
    categories.append({"id": cat_id, "name": name})
 
image_id = 1
ann_id   = 1

In [10]:
for cat_id, class_name in enumerate(class_names, start=1):
    class_dir = os.path.join(CONFIG["data_dir"], class_name)
    img_files = [
        f for f in sorted(os.listdir(class_dir))
        if f.lower().endswith(IMG_EXTS)
    ]

In [11]:
for fname in img_files:
        img_path = os.path.join(class_dir, fname)
 
        # Get image dimensions
        with Image.open(img_path) as img:
            w, h = img.size
 
        # Image entry — store relative path so it works across machines
        rel_path = os.path.join(class_name, fname)
        images.append({
            "id":        image_id,
            "file_name": rel_path,
            "width":     w,
            "height":    h,
        })

In [12]:
# Bounding box = full image [x, y, width, height] (COCO format)
annotations.append({
            "id":          ann_id,
            "image_id":    image_id,
            "category_id": cat_id,
            "bbox":        [0, 0, w, h],   # [x_min, y_min, width, height]
            "area":        w * h,
            "iscrowd":     0,
        })
 
image_id += 1
ann_id   += 1

In [13]:
coco_dict = {
    "info":        {"description": "Auto-generated from folder dataset"},
    "categories":  categories,
    "images":      images,
    "annotations": annotations,
}

In [14]:
import json

with open(OUTPUT_JSON, "w") as f:
    json.dump(coco_dict, f, indent=2)

In [15]:
print(f"\nDone. Saved {len(images)} images, {len(annotations)} annotations.")
print(f"Output: {OUTPUT_JSON}")
print("\nLabel map:")
for cat in categories:
    print(f"  {cat['name']!r}: {cat['id']}")
 


Done. Saved 2135 images, 1 annotations.
Output: TrashBox_train_set\annotations.json

Label map:
  'cardboard': 1
  'e-waste': 2
  'glass': 3
  'medical': 4
  'metal': 5
  'paper': 6
  'plastic': 7


In [16]:
import random

class COCODetectionDataset(Dataset):
    """
    Loads images and bounding boxes from a COCO-format JSON file.
    Images are stored in DATA_DIR/<class_name>/filename.jpg
    """
 
    def __init__(self, data_dir, annot_file, augment=False):
        self.data_dir = data_dir
        self.augment  = augment
 
        with open(annot_file) as f:
            coco = json.load(f)
 
        # Build id → category map
        self.cat_map = {cat["id"]: cat["name"] for cat in coco["categories"]}
 
        # Build image_id → annotations map
        ann_by_image = {}
        for ann in coco["annotations"]:
            iid = ann["image_id"]
            ann_by_image.setdefault(iid, []).append(ann)
 
        # Build flat list of samples
        self.samples = []
        for img_info in coco["images"]:
            iid   = img_info["id"]
            fpath = os.path.join(data_dir, img_info["file_name"])
            anns  = ann_by_image.get(iid, [])
            if not anns:
                continue
            self.samples.append((fpath, anns))
 
    def __len__(self):
        return len(self.samples)
 
    def __getitem__(self, idx):
        img_path, anns = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
 
        # Build boxes and labels
        boxes, labels = [], []
        for ann in anns:
            x, y, w, h = ann["bbox"]          # COCO: [x_min, y_min, w, h]
            xmin, ymin = x, y
            xmax, ymax = x + w, y + h
            # Skip degenerate boxes
            if xmax <= xmin or ymax <= ymin:
                continue
            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(ann["category_id"])  # 1-indexed (0 = background)
 
        if self.augment:
            image, boxes = self._augment(image, boxes)
 
        image  = TF.to_tensor(image)           # (C, H, W), float32 in [0,1]
        boxes  = torch.as_tensor(boxes,  dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)
 
        target = {
            "boxes":    boxes,
            "labels":   labels,
            "image_id": torch.tensor([idx]),
            "area":     (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0]),
            "iscrowd":  torch.zeros(len(labels), dtype=torch.int64),
        }
        return image, target
 
    def _augment(self, image, boxes):
        """Horizontal flip augmentation — safe for bounding boxes."""
        if random.random() > 0.5:
            w = image.width
            image = TF.hflip(image)
            boxes = [[w - xmax, ymin, w - xmin, ymax]
                     for xmin, ymin, xmax, ymax in boxes]
        return image, boxes
 
 
def collate_fn(batch):
    return tuple(zip(*batch))
 

In [17]:
import os
import random
from torch.utils.data import DataLoader, Subset

# Set seed for reproducibility
random.seed(CONFIG["seed"])

# Path to the generated COCO JSON file
ANNOT_FILE = os.path.join(CONFIG["data_dir"], "annotations.json")

# Build datasets
full_train = COCODetectionDataset(
    CONFIG["data_dir"],
    annot_file=ANNOT_FILE,
    augment=True
)

full_val = COCODetectionDataset(
    CONFIG["data_dir"],
    annot_file=ANNOT_FILE,
    augment=False
)

# Split indices
n = len(full_train)
indices = list(range(n))
random.shuffle(indices)

split = int(n * CONFIG["val_split"])
val_idx = indices[:split]
train_idx = indices[split:]

# Create subsets
train_set = Subset(full_train, train_idx)
val_set = Subset(full_val, val_idx)

# Data loaders
train_loader = DataLoader(
    train_set,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=0,
    collate_fn=collate_fn,
    pin_memory=(CONFIG["device"] == "cuda")
)

val_loader = DataLoader(
    val_set,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn,
    pin_memory=(CONFIG["device"] == "cuda")
)

print(f"Train: {len(train_set)} | Val: {len(val_set)}")
print(f"Using annotation file: {ANNOT_FILE}")

Train: 1815 | Val: 320
Using annotation file: TrashBox_train_set\annotations.json


In [18]:
#The model
def build_model(num_classes):
    # num_classes + 1 because torchvision counts background as class 0
    model = fasterrcnn_resnet50_fpn(weights="DEFAULT")
    
    for param in model.backbone.parameters():
        param.requires_grad = False
    
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes + 1)
    return model
 
model = build_model(CONFIG["num_classes"]).to(CONFIG["device"])
 

In [19]:
#optimizer and scheduler
params    = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=CONFIG["learning_rate"], momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

In [20]:
#training loop
def train_one_epoch(model, loader, optimizer, device, epoch, total_epochs):
    model.train()
    total_loss = 0.0

    print(f"\nStarting Epoch {epoch}/{total_epochs} [Train]")
    progress_bar = tqdm(loader, desc=f"Epoch {epoch:02d}/{total_epochs} [Train]")

    for batch_idx, (images, targets) in enumerate(progress_bar, start=1):
        print(f"Loaded train batch {batch_idx}/{len(loader)}")

        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        print(f"Running forward pass for batch {batch_idx}...")
        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        progress_bar.set_postfix(loss=f"{loss.item():.4f}")

        print(f"Finished batch {batch_idx}/{len(loader)} | Loss: {loss.item():.4f}")

    return total_loss / len(loader)


def validate(model, loader, device, epoch, total_epochs):
    model.train()
    total_loss = 0.0

    print(f"\nStarting Epoch {epoch}/{total_epochs} [Val]")
    progress_bar = tqdm(loader, desc=f"Epoch {epoch:02d}/{total_epochs} [Val]")

    with torch.no_grad():
        for batch_idx, (images, targets) in enumerate(progress_bar, start=1):
            print(f"Loaded val batch {batch_idx}/{len(loader)}")

            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            print(f"Running validation forward pass for batch {batch_idx}...")
            loss_dict = model(images, targets)
            loss = sum(loss_dict.values())

            total_loss += loss.item()
            progress_bar.set_postfix(loss=f"{loss.item():.4f}")

            print(f"Finished val batch {batch_idx}/{len(loader)} | Loss: {loss.item():.4f}")

    return total_loss / len(loader)

In [21]:
print("Device:", CONFIG["device"])
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))

Device: cpu
Train batches: 1815
Val batches: 320


In [22]:
print("Device:", CONFIG["device"])
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))

Device: cpu
Train batches: 1815
Val batches: 320


In [ ]:
#the training loop with early stopping
import torchvision.transforms.functional as TF

best_val_loss    = np.inf
patience_counter = 0
PATIENCE = 8
train_losses = []
val_losses = []
 
for epoch in range(1, CONFIG["num_epochs"] + 1):
    train_loss = train_one_epoch(
    model,
    train_loader,
    optimizer,
    CONFIG["device"],
    epoch,
    CONFIG["num_epochs"]
)
    val_loss = validate(
    model,
    val_loader,
    CONFIG["device"],
    epoch,
    CONFIG["num_epochs"]
)
    scheduler.step()
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
 
    print(f"Epoch {epoch:02d}/{CONFIG['num_epochs']} | "
      f"Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f}")
 
    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "best_fasterrcnn.pth")
        print("  → Saved best model")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break
 
print(f"\nTraining complete. Best val loss: {best_val_loss:.4f}")


Starting Epoch 1/10 [Train]


Epoch 01/10 [Train]:   0%|          | 0/1815 [00:00<?, ?it/s]

Loaded train batch 1/1815
Running forward pass for batch 1...
Finished batch 1/1815 | Loss: 4.0968
Loaded train batch 2/1815
Running forward pass for batch 2...
Finished batch 2/1815 | Loss: 2.9150
Loaded train batch 3/1815
Running forward pass for batch 3...
Finished batch 3/1815 | Loss: 5.8485
Loaded train batch 4/1815
Running forward pass for batch 4...
Finished batch 4/1815 | Loss: 2.7020
Loaded train batch 5/1815
Running forward pass for batch 5...
Finished batch 5/1815 | Loss: 2.2561
Loaded train batch 6/1815
Running forward pass for batch 6...
Finished batch 6/1815 | Loss: 2.0772
Loaded train batch 7/1815
Running forward pass for batch 7...
Finished batch 7/1815 | Loss: 1.9985
Loaded train batch 8/1815
Running forward pass for batch 8...
Finished batch 8/1815 | Loss: 2.6999
Loaded train batch 9/1815
Running forward pass for batch 9...
Finished batch 9/1815 | Loss: 2.0852
Loaded train batch 10/1815
Running forward pass for batch 10...
Finished batch 10/1815 | Loss: 1.5582
Loaded 

Epoch 01/10 [Val]:   0%|          | 0/320 [00:00<?, ?it/s]

Loaded val batch 1/320
Running validation forward pass for batch 1...
Finished val batch 1/320 | Loss: 0.0866
Loaded val batch 2/320
Running validation forward pass for batch 2...
Finished val batch 2/320 | Loss: 0.0622
Loaded val batch 3/320
Running validation forward pass for batch 3...
Finished val batch 3/320 | Loss: 0.2579
Loaded val batch 4/320
Running validation forward pass for batch 4...
Finished val batch 4/320 | Loss: 0.0698
Loaded val batch 5/320
Running validation forward pass for batch 5...
Finished val batch 5/320 | Loss: 0.1087
Loaded val batch 6/320
Running validation forward pass for batch 6...
Finished val batch 6/320 | Loss: 0.1693
Loaded val batch 7/320
Running validation forward pass for batch 7...
Finished val batch 7/320 | Loss: 0.0461
Loaded val batch 8/320
Running validation forward pass for batch 8...
Finished val batch 8/320 | Loss: 0.5523
Loaded val batch 9/320
Running validation forward pass for batch 9...
Finished val batch 9/320 | Loss: 0.1609
Loaded val

...

Loaded train batch 1/1815
Running forward pass for batch 1...
Finished batch 1/1815 | Loss: 0.2537
Loaded train batch 2/1815
Running forward pass for batch 2...
Finished batch 2/1815 | Loss: 0.1206
Loaded train batch 3/1815
Running forward pass for batch 3...
Finished batch 3/1815 | Loss: 0.0962
Loaded train batch 4/1815
Running forward pass for batch 4...
Finished batch 4/1815 | Loss: 0.3360
Loaded train batch 5/1815
Running forward pass for batch 5...
Finished batch 5/1815 | Loss: 0.0450
Loaded train batch 6/1815
Running forward pass for batch 6...
Finished batch 6/1815 | Loss: 0.0647
Loaded train batch 7/1815
Running forward pass for batch 7...
Finished batch 7/1815 | Loss: 0.2066
Loaded train batch 8/1815
Running forward pass for batch 8...
Finished batch 8/1815 | Loss: 0.1316
Loaded train batch 9/1815
Running forward pass for batch 9...
Finished batch 9/1815 | Loss: 0.8005
Loaded train batch 10/1815
Running forward pass for batch 10...
Finished batch 10/1815 | Loss: 0.5553
Loaded 

Epoch 03/10 [Val]:   0%|          | 0/320 [00:00<?, ?it/s]

Loaded val batch 1/320
Running validation forward pass for batch 1...
Finished val batch 1/320 | Loss: 0.0770
Loaded val batch 2/320
Running validation forward pass for batch 2...
Finished val batch 2/320 | Loss: 0.0421
Loaded val batch 3/320
Running validation forward pass for batch 3...
Finished val batch 3/320 | Loss: 0.2402
Loaded val batch 4/320
Running validation forward pass for batch 4...
Finished val batch 4/320 | Loss: 0.0574
Loaded val batch 5/320
Running validation forward pass for batch 5...
Finished val batch 5/320 | Loss: 0.1044
Loaded val batch 6/320
Running validation forward pass for batch 6...
Finished val batch 6/320 | Loss: 0.1832
Loaded val batch 7/320
Running validation forward pass for batch 7...
Finished val batch 7/320 | Loss: 0.0517
Loaded val batch 8/320
Running validation forward pass for batch 8...
Finished val batch 8/320 | Loss: 0.5257
Loaded val batch 9/320
Running validation forward pass for batch 9...
Finished val batch 9/320 | Loss: 0.1388
Loaded val

Epoch 04/10 [Train]:   0%|          | 0/1815 [00:00<?, ?it/s]

Loaded train batch 1/1815
Running forward pass for batch 1...
Finished batch 1/1815 | Loss: 0.1670
Loaded train batch 2/1815
Running forward pass for batch 2...
Finished batch 2/1815 | Loss: 0.7189
Loaded train batch 3/1815
Running forward pass for batch 3...
Finished batch 3/1815 | Loss: 0.1361
Loaded train batch 4/1815
Running forward pass for batch 4...
Finished batch 4/1815 | Loss: 0.2503
Loaded train batch 5/1815
Running forward pass for batch 5...
Finished batch 5/1815 | Loss: 0.1636
Loaded train batch 6/1815
Running forward pass for batch 6...
Finished batch 6/1815 | Loss: 0.1503
Loaded train batch 7/1815
Running forward pass for batch 7...
Finished batch 7/1815 | Loss: 0.4221
Loaded train batch 8/1815
Running forward pass for batch 8...
Finished batch 8/1815 | Loss: 0.2537
Loaded train batch 9/1815
Running forward pass for batch 9...
Finished batch 9/1815 | Loss: 0.0638
Loaded train batch 10/1815
Running forward pass for batch 10...
Finished batch 10/1815 | Loss: 0.0383
Loaded 

In [ ]:
#The inference
def predict(model, img_path, device, score_threshold=0.5):
    model.eval()
 
    with open(ANNOT_FILE) as f:
        coco = json.load(f)
    id_to_label = {cat["id"]: cat["name"] for cat in coco["categories"]}
 
    image  = Image.open(img_path).convert("RGB")
    tensor = TF.to_tensor(image).unsqueeze(0).to(device)
 
    with torch.no_grad():
        preds = model(tensor)[0]
 
    keep   = preds["scores"] >= score_threshold
    boxes  = preds["boxes"][keep].cpu().numpy()
    labels = preds["labels"][keep].cpu().numpy()
    scores = preds["scores"][keep].cpu().numpy()
 
    results = []
    for box, label, score in zip(boxes, labels, scores):
        results.append({
            "label": id_to_label.get(int(label), "unknown"),
            "score": round(float(score), 3),
            "box":   box.tolist(),    # [xmin, ymin, xmax, ymax]
        })
    return results

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.show()